# XGBoost Ablation: Remove Worst Concentration + Patent Hotspot

This notebook removes the single worst `Concentration + Patent` hotspot from the XGBoost by-gene investigation, then retrains and re-analyzes the model with the same frozen by-gene+mRNA setup.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Build Current Pipeline Data

This ablation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`. It also keeps the frozen XGBoost by-gene+mRNA Optuna hyperparameters from the earlier baseline notebook so that the only change is the row removal.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)

enriched_df["patent_group"] = enriched_df.get(
    "patent_ID", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
enriched_df["Authorization_status"] = enriched_df.get(
    "Authorization_status", pd.Series(index=enriched_df.index, dtype=object)
).fillna("UNKNOWN")
enriched_df["Title"] = enriched_df.get(
    "Title", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## What This Notebook Removes

Removed hotspot: `US20220380773A1 @ 20 nM`

Why: in the by-gene XGBoost investigation, this was the worst large-sample `Concentration + Patent` hotspot, with strongly negative Spearman and high error.

Exact row from the investigation table:
- `Concentration_nM = 20.0`
- `patent_group = US20220380773A1`
- `n_samples = 168`
- `spearman = -0.556349`
- `mae = 38.583055`
- `mean_true = 41.410714`
- `mean_pred = 57.806583`

This makes it a strong candidate for source-specific transfer failure or protocol heterogeneity.


In [3]:
combo_mask = (
    (enriched_df["patent_group"] == "US20220380773A1")
    & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False))
)
removed_df = enriched_df.loc[combo_mask].copy()
filtered_df = enriched_df.loc[~combo_mask].reset_index(drop=True)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_df)),
    "rows_remaining": int(len(filtered_df)),
    "pct_removed": float(len(removed_df) / len(enriched_df)),
    "removed_hotspot": "US20220380773A1 @ 20 nM",
}])
removal_summary

X, groups, y = pipeline.prepare_for_classical_ml(
    filtered_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)
mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = filtered_df.loc[mask].reset_index(drop=True).copy()

print("Filtered dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))


Feature matrix X shape: (35274, 1425), target y shape: (35274,)
Filtered dataframe shape: (35274, 40)
Feature matrix shape: (35274, 1425)
Target shape: (35274,)
Unique genes: 54


## Helper Functions

In [4]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## Refit And Re-Analyze

The model is retrained from scratch after the hotspot removal, then evaluated again with the same by-gene out-of-fold procedure used in the main XGBoost investigation notebook.

In [5]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
baseline_metrics = pd.DataFrame([{'spearman': 0.45886, 'pearson': 0.453422, 'rmse': 31.421689, 'mae': 24.568816}])
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=30, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
comparison = pd.concat([baseline_metrics.assign(run="baseline_by_gene_mrna"), metrics_df.assign(run="ablation")], ignore_index=True)
delta_vs_baseline = metrics_df.iloc[0] - baseline_metrics.iloc[0]
delta_df = pd.DataFrame({"metric": delta_vs_baseline.index, "delta_vs_baseline": delta_vs_baseline.values})
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,24006,11268,54,16
1,2,24010,11264,54,16
2,3,24046,11228,54,16


## Fold Summary

## Baseline Comparison

In [6]:
comparison

,spearman,pearson,rmse,mae,run
0,0.458860,0.453422,31.421689,24.568816,baseline_by_gene_mrna
1,0.463177,0.460762,31.217869,24.219239,ablation


## Metric Delta Vs Baseline

In [7]:
delta_df

,metric,delta_vs_baseline
0,spearman,0.004317
1,pearson,0.007340
2,rmse,-0.203820
3,mae,-0.349577


## Overall Metrics

In [8]:
metrics_df

,spearman,pearson,rmse,mae
0,0.463177,0.460762,31.217869,24.219239


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations after the ablation.

In [9]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,15.00000,35,-0.725547,25.468775,86.038000,60.569233
1,3.00000,35,-0.639757,16.549227,71.316857,55.515759
2,0.00030,40,-0.176675,11.944686,5.699000,-1.314412
3,0.30000,145,-0.151104,21.537396,43.381517,37.268448
4,0.00300,64,-0.091263,15.022998,3.433438,-1.744882
5,100.00000,1038,-0.026222,30.212386,51.019336,57.066483
6,0.00910,48,0.021227,19.646860,12.089167,9.509524
7,2.22220,49,0.026792,13.901296,78.403469,71.821381
8,0.00064,113,0.033610,9.908887,-1.450796,0.166238
9,0.01000,198,0.078025,23.492167,36.725051,20.689024


## Concentration + Gene Hotspots

In [10]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,15.00000,HSD17B13,35,-0.725547,25.468775,86.038000,60.569233
1,3.00000,HSD17B13,35,-0.639757,16.549227,71.316857,55.515759
2,5.00000,HSD17B13,52,-0.409282,35.770653,34.843462,55.083401
3,0.74100,HSD17B13,16,-0.397059,29.996849,85.943750,58.873291
4,2.22200,HSD17B13,16,-0.388235,27.239138,87.950000,63.581287
5,0.08200,HSD17B13,16,-0.317647,37.276682,63.806250,31.209419
6,0.24700,HSD17B13,16,-0.311994,33.071091,78.275000,50.376217
7,5.00000,AGT,26,-0.299880,23.182562,70.858077,50.435848
8,0.02700,HSD17B13,16,-0.267647,32.283803,42.193750,11.519770
9,0.00300,LPA,22,-0.259176,15.041248,-1.755455,0.510308


## Concentration + Patent Hotspots

In [11]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,15.0000,WO2023091644A2,35,-0.725547,25.468775,86.038000,60.569233
1,3.0000,WO2023091644A2,35,-0.639757,16.549227,71.316857,55.515759
2,10.0000,CN112313335A,182,-0.445131,57.374968,92.877308,35.502338
3,0.1000,CN112313335A,184,-0.412402,85.827407,98.657065,12.829659
4,0.7410,CN116940681A,16,-0.397059,29.996849,85.943750,58.873291
5,2.2220,CN116940681A,16,-0.388235,27.239138,87.950000,63.581287
6,0.0003,CN116940681A,16,-0.365243,9.372888,5.756250,-0.668369
7,0.1000,WO2022089486A1,20,-0.326684,38.381125,62.100000,26.534708
8,0.0820,CN116940681A,16,-0.317647,37.276682,63.806250,31.209419
9,100.0000,CN109957567B,33,-0.312265,30.674256,64.969697,52.290455


## Experimental Feature Issues

In [12]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Primary mouse hepatocytes,312,56.108250,66.830060,-0.125206,25.020673,38.794361
1,Non-human hepatocytes,188,30.380352,36.569140,0.117474,22.580851,52.272430
2,Primary human hepatocytes,2924,25.798943,31.401914,0.154711,50.963902,47.106777
3,Primary Cynomolgus Monkey Hepatocytes,4231,26.574108,33.930726,0.266978,34.441572,42.737572
4,Human iPSC-derived cortical neurons,198,24.638885,28.798719,0.384232,42.520202,43.369877
5,Hep3B,11834,28.004667,35.359476,0.385665,42.244836,38.134830
6,HepG2,1629,20.838224,27.509017,0.467040,40.621050,48.801933
7,Huh7,1542,21.292563,27.208919,0.519934,40.970402,37.974358
8,H1299 Cells,1464,12.315442,15.494601,0.601476,68.351776,68.689560
9,HeLa Cells,714,14.107403,17.702157,0.611051,54.726912,58.904118


## Concentration Issue Summary

In [13]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,15.00000,35,25.468775,26.319141,-0.725547,86.038000,60.569233
1,3.00000,35,16.549227,19.491058,-0.639757,71.316857,55.515759
2,0.00030,40,11.944686,14.294201,-0.176675,5.699000,-1.314412
3,0.30000,145,21.537396,27.515205,-0.151104,43.381517,37.268448
4,0.00300,64,15.022998,18.261618,-0.091263,3.433438,-1.744882
5,100.00000,1038,30.212386,37.474336,-0.026222,51.019336,57.066483
6,0.00910,48,19.646860,23.266682,0.021227,12.089167,9.509524
7,2.22220,49,13.901296,17.309677,0.026792,78.403469,71.821381
8,0.00064,113,9.908887,12.502002,0.033610,-1.450796,0.166238
9,0.01000,198,23.492167,30.552902,0.078025,36.725051,20.689024


## Time Issue Summary

In [14]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,40.0,195,16.778220,20.945809,0.257990,27.533333,20.844625
1,168.0,198,24.638885,28.798719,0.384232,42.520202,43.369877
2,24.0,23812,26.360626,33.678979,0.407862,41.671578,40.559013
3,72.0,1388,20.536764,25.488125,0.433991,15.815331,29.313305
4,48.0,8151,18.762299,24.089424,0.566773,48.728591,50.053288


## Patent Issue Summary

In [15]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,CN112313335A,366,71.678927,73.925569,-0.767038,95.782978,24.104052
1,CN109957567B,39,29.358298,37.067353,-0.226019,63.641026,49.509281
2,CN113980966A,57,38.112798,43.934262,-0.218814,83.438596,46.831783
3,WO2023213284A1,48,31.589366,33.447624,-0.067694,86.482083,56.356903
4,CN117210468A,223,21.003110,26.657927,-0.030114,25.394619,25.877321
5,WO2023091644A2,1439,25.490318,31.001235,0.001820,48.175136,48.710865
6,WO2022121959A1,100,41.764175,49.266545,0.051621,64.982500,45.208241
7,CN116732034A,30,37.636268,40.082226,0.083426,83.838000,46.201733
8,CN116801886A,267,31.301968,37.957774,0.098668,50.756891,42.310093
9,TW202321444A,188,30.380352,36.569140,0.117474,22.580851,52.272430


## Authorization Issue Summary

In [16]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,223,21.003110,26.657927,-0.030114,25.394619,25.877321
1,Substantive Examination,15968,26.634975,33.424987,0.469081,46.312121,36.600140
2,Unknown Status,12457,23.796319,31.217308,0.488220,33.375278,45.883369
3,Withdrawn,1079,21.580732,26.624494,0.574888,30.078591,32.059822
4,Granted,1700,21.769546,28.072533,0.583571,49.092965,46.631039
5,UNKNOWN,2333,13.255875,16.799303,0.647351,63.775570,65.137558


## Prediction Preview

In [17]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,LPA,Hep3B,1.0,24.0,CN108368506A,-1.2,2.536098,-3.736098,3.736098
1,LPA,Hep3B,1.0,24.0,CN108368506A,22.4,-19.362045,41.762045,41.762045
2,LPA,Hep3B,1.0,24.0,CN108368506A,29.2,25.475090,3.724910,3.724910
3,LPA,Hep3B,1.0,24.0,CN108368506A,55.9,26.887688,29.012312,29.012312
4,LPA,Hep3B,1.0,24.0,CN108368506A,75.8,19.325972,56.474028,56.474028
5,LPA,Hep3B,1.0,24.0,CN108368506A,83.4,44.050579,39.349421,39.349421
6,LPA,Hep3B,1.0,24.0,CN108368506A,29.8,-0.928164,30.728164,30.728164
7,LPA,Hep3B,1.0,24.0,CN108368506A,71.0,14.263166,56.736834,56.736834
8,LPA,Hep3B,1.0,24.0,CN108368506A,17.5,24.741076,-7.241076,7.241076
9,LPA,Hep3B,1.0,24.0,CN108368506A,34.6,32.397720,2.202280,2.202280


## Interpretation Notes

- Removing the single patent hotspot `US20220380773A1 @ 20 nM` produces a real but modest improvement over the original by-gene XGBoost baseline.
- Overall metrics improve from `Spearman 0.4589` to `0.4632`, `Pearson 0.4534` to `0.4608`, `RMSE 31.42` to `31.22`, and `MAE 24.57` to `24.22`.
- This means the hotspot was genuinely harmful, but it was not the dominant source of failure by itself.
- After removing that patent slice, the main remaining problem pattern shifts strongly toward `HSD17B13` across several concentrations. In the post-ablation `Concentration + Gene` table, the worst rows are now `HSD17B13 @ 15 nM`, `3 nM`, `5 nM`, `0.741`, `2.222`, `0.082`, `0.247`, and `0.300 nM`.
- The post-ablation `Concentration + Patent` table also shows that `CN112313335A` remains a major warning source, especially at `10 nM` and `0.1 nM`, with very poor Spearman and very large error. `WO2023091644A2` also appears repeatedly in the remaining worst slices.
- Cell-type issues remain broadly similar after the ablation. `Primary mouse hepatocytes` is still the clearest cell-context warning sign, with negative Spearman and very high error, so this ablation did not solve that transfer problem.
- Time of administration still does not look like the main poisoning axis. `72h` is not the worst signal here either.
- So this notebook supports the idea that localized patent heterogeneity matters, but also shows that removing one bad patent hotspot is not sufficient to clean up the full transfer problem.

## Conclusions And Next Step

- `US20220380773A1 @ 20 nM` looks like a valid targeted filtering candidate because removing it helps every top-level metric.
- However, the gain is small enough that it should probably be treated as one member of a broader hotspot set rather than the single key fix.
- The strongest next targets suggested by this notebook are the remaining high-impact patent/source hotspots and the repeated `HSD17B13` concentration failures.
- The best next step is to compare this notebook directly against the gene-hotspot ablation notebook and then prioritize whichever axis gives the larger and more stable gain.
- Based on the saved results, we should probably keep `US20220380773A1 @ 20 nM` on the candidate-removal list, but not stop there.
